# Projet B3 · L'analyste automatique (avec un humain dans la boucle) · ⭐⭐⭐

**Le problème** : devant un CSV inconnu, on refait toujours les mêmes gestes — regarder les colonnes, compter les manquants, sortir des statistiques, choisir quoi tracer. C'est long, et c'est exactement ce qu'une machine sait faire.
**Ce qu'on construit** : un agent qui fait ces gestes tout seul, écrit un mini-rapport en français, **répond à une question posée en français** en écrivant lui-même la requête, et **propose** trois graphiques — mais il ne trace rien sans ton accord.
**Livrable** : ce notebook complété sur **un CSV de ton choix** : le rapport automatique, les 3 graphiques proposés dont au moins 2 acceptés, le journal de tes décisions et la fiche projet finale.

**Comment l'utiliser**
- Google Colab, rien à installer. Exécute chaque cellule avec `Maj + Entrée`.
- L'interrupteur `USE_MODEL` est en tête : `False` = mode démo, tout tourne **sans GPU et sans clé** (le faux modèle écrit quand même les requêtes et rédige le rapport à partir des chiffres calculés par **ton** code) ; `True` = le petit modèle Qwen tourne dans Colab. Le reste du notebook ne change pas.
- Les cellules **« À toi »** sont des exercices : elles s'exécutent telles quelles, la vérification affiche ✅ ou ❌, la solution est cachée juste en dessous — essaie avant de l'ouvrir.
- **Pas d'`input()`** : la validation humaine se fait en éditant une liste `DECISIONS` puis en relançant. Un notebook doit pouvoir se rejouer de haut en bas sans qu'on tape quoi que ce soit.
- Ce projet prolonge la [séance 5 · Analyser et raconter](../../../seances/seance-05-analyser-raconter/) et la [séance 12 · Agents](../../../seances/seance-12-agents-et-projet-final/).

## 0. Préparation

La cellule `llm(messages)` des séances 9 à 12, avec les trois métiers de ce projet écrits dans le faux modèle : **écrire une requête**, **rédiger le rapport**, **commenter un résultat**. La cellule suivante contient le détail de ces trois branches — du bricolage assumé, qui permet de développer et de vérifier tout le notebook sans GPU ; avec `USE_MODEL = True`, elles ne servent jamais. Lance les deux une fois.

In [ ]:
USE_MODEL = True   # ← mets False pour tester le notebook sans modèle (réponses factices, sans GPU)

import json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Mode démo : un faux LLM qui répond sans réseau ni GPU ----------
def llm_factice(messages):
    """Trois métiers seulement — écrire une requête, rédiger un rapport, commenter un résultat.
    Le détail de chaque branche est dans la cellule suivante."""
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
    if "Écris la requête" in systeme:
        return _requete_factice(systeme, messages[-1]["content"])
    if "Faits :" in systeme:
        return _rediger_factice(systeme)
    return "Je n'ai pas d'outil pour cette demande : donne-moi des faits ou une question sur le fichier."

# ---------- Le vrai modèle : petit modèle ouvert, gratuit, sans clé ----------

if USE_MODEL:
    %pip install -q transformers accelerate
    from transformers import pipeline
    _pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def llm(messages, max_new_tokens=150, temperature=0.7):
    """Envoie une liste de messages au modèle et renvoie sa réponse (du texte)."""
    if not USE_MODEL:
        return llm_factice(messages)
    if temperature == 0:      # température 0 = toujours la réponse la plus probable
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    return sortie[0]["generated_text"][-1]["content"].strip()

print("Modèle prêt :", "Qwen2.5-0.5B-Instruct" if USE_MODEL else "mode démo (llm_factice)")

In [ ]:
# Le détail des trois branches du faux modèle, à lire une fois et à oublier ensuite :
# du bricolage assumé, qui permet de tout développer et vérifier sans GPU.
# Avec USE_MODEL = True, rien de ce qui suit ne sert.
MOTS_FR = {"poids": "mass", "lourd": "mass", "masse": "mass", "espèce": "species", "île": "island",
           "pourboire": "tip", "addition": "total_bill", "jour": "day", "attaque": "attack",
           "défense": "defense", "vitesse": "speed", "génération": "generation", "sexe": "sex",
           "bec": "bill", "nageoire": "flipper", "fumeur": "smoker", "type": "type"}

def _liste(systeme, etiquette):
    """Relit une liste de colonnes écrite dans le prompt système."""
    if etiquette not in systeme:
        return []
    return [c.strip() for c in systeme.split(etiquette, 1)[1].splitlines()[0].split(",") if c.strip()]

def _colonne(texte, colonnes):
    """La colonne visée par un bout de phrase : un nom cité tel quel, sinon un petit dictionnaire français."""
    bas = texte.lower()
    pistes = [c.lower() for c in colonnes if c.lower() in bas] + [v for m, v in MOTS_FR.items() if m in bas]
    for piste in pistes:
        for c in colonnes:
            if piste in c.lower():
                return c
    return colonnes[0] if colonnes else ""

print("Repérage des colonnes prêt :", len(MOTS_FR), "mots français reconnus")

In [ ]:
def _requete_factice(systeme, question):
    """Écrit UNE commande du mini-langage, toujours la même pour une même question."""
    num, cat = _liste(systeme, "Colonnes numériques :"), _liste(systeme, "Colonnes catégorielles :")
    q, seuil = question.lower(), re.search(r"\d+", question)
    if "plus lourd" in q:                     # ⚠️ le piège volontaire de la section 4.3
        return "moyenne(%s)" % _colonne(q, num)
    if " par " in q or "chaque" in q:
        return "moyenne_par(%s, %s)" % (_colonne(q, cat), _colonne(q, num))
    if "fréquent" in q or "réparti" in q or "courant" in q:
        return "top(%s, 5)" % _colonne(q, cat)
    if "combien" in q:
        return "compte(%s)" % _colonne(q, num or cat)
    if "médian" in q:
        return "mediane(%s)" % _colonne(q, num)
    if seuil and ("dépasse" in q or "plus de" in q or "au-dessus" in q):
        morceaux = re.split(r"\bquand\b|\bsi\b|\blorsque\b", q, maxsplit=1)
        return "filtre(%s, >, %s) puis moyenne(%s)" % (_colonne(morceaux[-1], num), seuil.group(), _colonne(morceaux[0], num))
    return "moyenne(%s)" % _colonne(q, num)

def _rediger_factice(systeme):
    """Met en phrases les chiffres du dictionnaire de faits — et n'en invente aucun."""
    faits = json.loads(systeme.split("Faits :", 1)[1].strip())
    if "classement" in faits:
        nom, valeur = faits["classement"][0]
        return "En tête : %s, avec %s — sur %s groupes comparés." % (nom, valeur, faits["nb_groupes"])
    if "valeur" in faits:
        return "Réponse : %s (obtenue avec la commande %s)." % (faits["valeur"], faits["commande"])
    lignes = ["Le fichier compte %s lignes et %s colonnes." % (faits["lignes"], faits["colonnes"]),
              "Il y a %s colonnes numériques et %s colonnes de texte." % (faits["numeriques"], faits["categorielles"]),
              "Valeurs manquantes : %s au total, dont %s pour « %s »." % (faits["manquants_total"], faits["pire_manquants"][1], faits["pire_manquants"][0]),
              "Lignes en double : %s." % faits["doublons"]]
    for nom, s in faits["colonnes_detail"].items():
        if "moyenne" in s:
            lignes.append("« %s » : moyenne %s, médiane %s, de %s à %s." % (nom, s["moyenne"], s["mediane"], s["min"], s["max"]))
    lignes.append("Alertes relevées automatiquement : %s." % faits["nb_alertes"])
    return "\n".join(lignes)

print("Branches du mode démo prêtes.")

Puis les deux outils du projet : `demander()` (une question + un prompt système) et le helper `verifier` qui affiche ✅ / ❌ sans jamais planter.

*Si le formateur a donné une clé d'API, la variante des séances 10 à 12 se branche ici sans rien changer d'autre : il suffit de redéfinir `llm(messages)`, tout le notebook ne passe que par elle.*

In [ ]:
def demander(question, systeme="Tu es un analyste qui répond en français, en 2 phrases maximum."):
    return llm([{"role": "system", "content": systeme}, {"role": "user", "content": question}])

def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais planter. `condition` = un booléen, ou une fonction sans argument."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as e:
        ok = False
        print(f"   (erreur pendant la vérification : {type(e).__name__} : {e})")
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else "  → pas encore, relis l'énoncé et réessaie"))

print("Outils prêts.")

## 1. L'agent propose, l'humain dispose

Un agent d'analyse, c'est tentant : « voici mon fichier, dis-moi ce qu'il contient ». Le danger est ailleurs que là où on l'attend. Ce n'est pas qu'il se trompe de calcul — un `mean()` est un `mean()`. C'est qu'il **répond à une autre question que la tienne**, avec un code parfaitement correct et une phrase parfaitement fluide. Rien ne clignote en rouge.

D'où les trois règles de ce projet :

1. **L'agent ne calcule rien lui-même.** Il écrit une **requête** dans un mini-langage que *nous* avons défini, et c'est *notre* code qui l'exécute. Pas d'`exec`, pas d'`eval` : une liste blanche d'opérations, et un refus propre pour tout le reste.
2. **Aucun chiffre ne sort de la plume du modèle.** Tous les chiffres sont calculés par pandas, rangés dans un **dictionnaire de faits**, et le modèle ne fait que les mettre en phrases. On vérifie ensuite, chiffre par chiffre, qu'il n'en a pas inventé un.
3. **Rien n'est tracé sans accord.** L'agent *propose* trois graphiques ; une liste `DECISIONS` que tu édites dit `"oui"`, `"non"`, ou donne une consigne. Le journal garde la trace de tes choix.

Le plan en 4 étapes :

| Étape | Ce que fait l'agent | Ce que tu contrôles |
|---|---|---|
| **1. Décrire** | forme, types, manquants, cardinalité → un dictionnaire de faits | tu lis les faits, pas un texte |
| **2. Alerter** | statistiques selon le type, valeurs aberrantes, colonnes suspectes | tu vois la liste des alertes |
| **3. Rédiger et répondre** | un rapport de 10 lignes, puis des réponses à tes questions | tu vérifies chaque chiffre |
| **4. Proposer** | 3 graphiques justifiés par le type des colonnes | `DECISIONS` : oui / non / une consigne |

## 2. Les données : n'importe quel CSV

Trois fichiers publics sont prêts, chargés par URL. **Change la variable `JEU`** pour passer de l'un à l'autre : c'est le seul endroit du notebook où le fichier est nommé. Tout le reste doit marcher sans écrire un seul nom de colonne en dur — c'est la contrainte principale du projet.

Si le réseau manque (Colab hors ligne, pare-feu), un mini-jeu de secours écrit dans la cellule prend le relais : le notebook tourne quand même de bout en bout.

In [ ]:
JEUX = {
    "manchots":   "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv",
    "pourboires": "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv",
    "pokemon":    "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv",
}
JEU = "manchots"        # ← le seul endroit où le fichier est choisi (essaie "pourboires", "pokemon", ou ton URL)

SECOURS = {"species": ["Adelie"] * 4 + ["Gentoo"] * 4 + ["Chinstrap"] * 4,
           "island": ["Torgersen", "Biscoe", "Dream", "Biscoe"] * 3,
           "bill_length_mm": [39.1, 38.6, 40.3, 37.8, 46.1, 50.0, 48.7, 47.2, 46.5, 49.2, 50.6, 45.7],
           "bill_depth_mm": [18.7, 17.2, 18.0, 17.3, 13.2, 15.0, 14.1, 13.7, 17.9, 18.2, 19.4, 17.0],
           "flipper_length_mm": [181, 191, 195, 186, 211, 222, 210, 214, 192, 195, 199, 190],
           "body_mass_g": [3750, 3800, 3250, 3300, 4500, 5700, 4450, 4925, 3500, 3800, 3800, 3650],
           "sex": ["Male", "Female", "Female", None, "Female", "Male", "Female", "Male", "Female", "Male", "Male", None]}

def charger(nom):
    """Charge le CSV par URL ; sans réseau, replie sur le mini-jeu écrit ci-dessus."""
    try:
        return pd.read_csv(JEUX[nom])
    except Exception as e:
        print("réseau indisponible :", type(e).__name__, "→ repli sur le mini-jeu de secours")
        return pd.DataFrame(SECOURS)

df = charger(JEU)
print(JEU, ":", df.shape[0], "lignes ×", df.shape[1], "colonnes")
df.head()

### À toi · exercice 1 ⭐ · Numérique ou catégoriel ?

Tout le reste du notebook dépend de cette question : **de quel type est cette colonne ?** On n'a pas le droit de le savoir à l'avance — le fichier est inconnu.

Écris `types_de_colonnes(df)` : elle renvoie un dictionnaire `{nom de colonne: "numerique" ou "categorielle"}`, pour **n'importe quel** DataFrame.

Résultat attendu sur les manchots : `bill_length_mm` → `"numerique"`, `species` → `"categorielle"`.

<details><summary>Indice</summary>

`pd.api.types.is_numeric_dtype(df[c])` répond `True` ou `False` pour la colonne `c`. Une compréhension de dictionnaire sur `df.columns` suffit.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def types_de_colonnes(df):
    """{colonne: 'numerique' ou 'categorielle'} — aucun nom de colonne écrit en dur."""
    return {c: ("numerique" if pd.api.types.is_numeric_dtype(df[c]) else "categorielle") for c in df.columns}

print(types_de_colonnes(df))
```

</details>

In [ ]:
# À toi
def types_de_colonnes(df):
    return {}

print(types_de_colonnes(df))

In [ ]:
verifier("Exercice 1 · une entrée par colonne", lambda: len(types_de_colonnes(df)) == df.shape[1])
verifier("Exercice 1 · deux étiquettes seulement", lambda: set(types_de_colonnes(df).values()) <= {"numerique", "categorielle"})
verifier("Exercice 1 · marche sur un tableau quelconque",
         lambda: types_de_colonnes(pd.DataFrame({"zz": [1, 2], "aa": ["x", "y"]})) == {"zz": "numerique", "aa": "categorielle"})

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Replié comme une solution : il réinjecte l'implémentation de référence,
# pour que la suite du notebook fonctionne quand même.
if not types_de_colonnes(df):
    def types_de_colonnes(df):
        return {c: ("numerique" if pd.api.types.is_numeric_dtype(df[c]) else "categorielle") for c in df.columns}

### À toi · exercice 2 ⭐⭐ · `decrire(df)` renvoie des **faits**, pas du texte

C'est la pièce maîtresse. `decrire(df)` ne doit **rien rédiger** : elle renvoie un dictionnaire de faits bruts, que le modèle mettra en phrases plus tard. Séparer les deux, c'est ce qui garantit que les chiffres du rapport final viennent tous de pandas.

Écris `decrire(df)` avec exactement ces clés :

| clé | contenu |
|---|---|
| `lignes`, `colonnes` | deux entiers (la forme) |
| `noms` | la liste des noms de colonnes |
| `types` | `{colonne: le dtype écrit en toutes lettres}` |
| `manquants` | `{colonne: nombre de valeurs manquantes}` |
| `cardinalite` | `{colonne: nombre de valeurs différentes}` |
| `doublons` | le nombre de lignes strictement identiques |

Toutes les valeurs doivent être des types Python simples (`int`, `str`, `list`, `dict`) : le dictionnaire finira dans un `json.dumps`, et `numpy.int64` n'y passe pas.

<details><summary>Indice</summary>

`int(df[c].isna().sum())`, `int(df[c].nunique())`, `str(df[c].dtype)`, `int(df.duplicated().sum())`.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def decrire(df):
    """Un dictionnaire de faits sur n'importe quel DataFrame — aucun nom de colonne en dur."""
    return {"lignes": int(len(df)),
            "colonnes": int(df.shape[1]),
            "noms": list(df.columns),
            "types": {c: str(df[c].dtype) for c in df.columns},
            "manquants": {c: int(df[c].isna().sum()) for c in df.columns},
            "cardinalite": {c: int(df[c].nunique()) for c in df.columns},
            "doublons": int(df.duplicated().sum())}

faits_bruts = decrire(df)
print(json.dumps(faits_bruts, ensure_ascii=False, indent=1)[:400], "...")
```

</details>

In [ ]:
# À toi
def decrire(df):
    return {}

faits_bruts = decrire(df)
print(faits_bruts)

In [ ]:
CLES = {"lignes", "colonnes", "noms", "types", "manquants", "cardinalite", "doublons"}
verifier("Exercice 2 · les 7 clés sont là", lambda: CLES <= set(decrire(df)))
verifier("Exercice 2 · la forme est juste", lambda: (decrire(df)["lignes"], decrire(df)["colonnes"]) == df.shape)
verifier("Exercice 2 · aucun nom de colonne en dur",
         lambda: decrire(pd.DataFrame({"zz": [1, None, 1]}))["manquants"] == {"zz": 1})
verifier("Exercice 2 · convertible en JSON (pas de numpy)", lambda: isinstance(json.dumps(decrire(df)), str))

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Replié comme une solution : il réinjecte l'implémentation de référence,
# pour que la suite du notebook fonctionne quand même.
if not CLES <= set(decrire(df)):
    def decrire(df):
        return {"lignes": int(len(df)), "colonnes": int(df.shape[1]), "noms": list(df.columns),
                "types": {c: str(df[c].dtype) for c in df.columns},
                "manquants": {c: int(df[c].isna().sum()) for c in df.columns},
                "cardinalite": {c: int(df[c].nunique()) for c in df.columns},
                "doublons": int(df.duplicated().sum())}
    faits_bruts = decrire(df)

## 3. Les statistiques utiles, et les alertes

`df.describe()` sort une moyenne pour tout ce qui ressemble à un nombre — y compris pour un numéro de ligne ou un code postal, où elle ne veut rien dire. Un analyste choisit ses statistiques **selon le type de la colonne** : moyenne / médiane / écart-type / extrêmes pour une colonne numérique, top 5 des valeurs pour une colonne de texte.

### À toi · exercice 3 ⭐⭐ · Les statistiques du bon type

Écris `stats_colonne(df, col)` qui renvoie un dictionnaire :

- colonne **numérique** → `{"type": "numerique", "moyenne", "mediane", "ecart_type", "min", "max"}`, arrondis à 2 décimales ;
- colonne **catégorielle** → `{"type": "categorielle", "valeurs_differentes", "top"}` où `top` est une liste `[[valeur, effectif], ...]` des 5 valeurs les plus fréquentes ;
- colonne **entièrement vide** → `{"type": ..., "vide": True}` (sinon la moyenne vaut `NaN` et le rapport dira n'importe quoi).

Là encore : uniquement des types Python simples, pour le `json.dumps` de la section 4.

<details><summary>Indice</summary>

Commence par `serie = df[col].dropna()`, puis `if serie.empty: ...`. Pour le top : `serie.value_counts().head(5).items()`.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def stats_colonne(df, col):
    """Les statistiques qui ont un sens pour CE type de colonne."""
    numerique = pd.api.types.is_numeric_dtype(df[col])
    serie = df[col].dropna()
    if serie.empty:
        return {"type": "numerique" if numerique else "categorielle", "vide": True}
    if numerique:
        return {"type": "numerique", "moyenne": round(float(serie.mean()), 2),
                "mediane": round(float(serie.median()), 2), "ecart_type": round(float(serie.std()), 2),
                "min": round(float(serie.min()), 2), "max": round(float(serie.max()), 2)}
    return {"type": "categorielle", "valeurs_differentes": int(serie.nunique()),
            "top": [[str(v), int(n)] for v, n in serie.value_counts().head(5).items()]}

print(stats_colonne(df, faits_bruts["noms"][0]))
```

</details>

In [ ]:
# À toi
def stats_colonne(df, col):
    return {}

print(stats_colonne(df, faits_bruts["noms"][0]))

In [ ]:
_petit = pd.DataFrame({"n": [1.0, 3.0, 5.0], "t": ["a", "a", "b"], "vide": [None, None, None]})
verifier("Exercice 3 · numérique : moyenne juste", lambda: stats_colonne(_petit, "n")["moyenne"] == 3.0)
verifier("Exercice 3 · numérique : les 5 statistiques",
         lambda: {"moyenne", "mediane", "ecart_type", "min", "max"} <= set(stats_colonne(_petit, "n")))
verifier("Exercice 3 · catégorielle : le top des valeurs", lambda: stats_colonne(_petit, "t")["top"][0] == ["a", 2])
verifier("Exercice 3 · colonne vide signalée", lambda: stats_colonne(_petit, "vide").get("vide") is True)

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Replié comme une solution : il réinjecte l'implémentation de référence,
# pour que la suite du notebook fonctionne quand même.
if "moyenne" not in stats_colonne(_petit, "n"):
    def stats_colonne(df, col):
        numerique = pd.api.types.is_numeric_dtype(df[col])
        serie = df[col].dropna()
        if serie.empty:
            return {"type": "numerique" if numerique else "categorielle", "vide": True}
        if numerique:
            return {"type": "numerique", "moyenne": round(float(serie.mean()), 2),
                    "mediane": round(float(serie.median()), 2), "ecart_type": round(float(serie.std()), 2),
                    "min": round(float(serie.min()), 2), "max": round(float(serie.max()), 2)}
        return {"type": "categorielle", "valeurs_differentes": int(serie.nunique()),
                "top": [[str(v), int(n)] for v, n in serie.value_counts().head(5).items()]}

Un dictionnaire de statistiques ne se lit pas. La fonction ci-dessous le **lit en une phrase** — une phrase par colonne, choisie selon le type. C'est déjà un rapport, écrit sans modèle du tout : garde-le en tête quand tu compareras avec la version rédigée par le LLM. La cellule affiche d'abord les faits bruts de `decrire`, puis leur lecture.

In [ ]:
faits_bruts = decrire(df)
for nom in faits_bruts["noms"]:
    print(f"  {nom:20s} {faits_bruts['types'][nom]:>8s}  "
          f"{faits_bruts['manquants'][nom]:>4d} manquants  "
          f"{faits_bruts['cardinalite'][nom]:>5d} valeurs différentes")
print()

def en_une_phrase(nom, s):
    """Lit un dictionnaire de statistiques en une phrase française."""
    if s.get("vide"):
        return f"« {nom} » : colonne entièrement vide, rien à dire."
    if s["type"] == "numerique":
        return (f"« {nom} » : en moyenne {s['moyenne']}, la moitié des lignes sous {s['mediane']}, "
                f"de {s['min']} à {s['max']} (écart-type {s['ecart_type']}).")
    tete = ", ".join(f"{v} ({n})" for v, n in s["top"][:3])
    return f"« {nom} » : {s['valeurs_differentes']} valeurs différentes, les plus fréquentes sont {tete}."

for nom in faits_bruts["noms"]:
    print(en_une_phrase(nom, stats_colonne(df, nom)))

Reste ce qu'un analyste voit **du coin de l'œil** et qu'aucune moyenne ne montre : une colonne qui ne contient qu'une seule valeur (inutile), une colonne qui en contient autant que de lignes (c'est un identifiant, pas une donnée), des lignes en double, et les **valeurs aberrantes**.

La règle des **1,5 × écart interquartile** : on appelle Q1 la valeur sous laquelle se trouvent 25 % des lignes, Q3 celle sous laquelle se trouvent 75 %, et EIQ = Q3 − Q1. Tout ce qui sort de l'intervalle [Q1 − 1,5 × EIQ ; Q3 + 1,5 × EIQ] est signalé. Ce n'est pas une erreur — c'est une valeur à **regarder**.

In [ ]:
def valeurs_aberrantes(df, col):
    """Les valeurs hors de [Q1 - 1,5 EIQ ; Q3 + 1,5 EIQ] (règle de Tukey)."""
    serie = df[col].dropna().astype("float64")     # astype : les colonnes vrai/faux passent aussi
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    eiq = q3 - q1
    return serie[(serie < q1 - 1.5 * eiq) | (serie > q3 + 1.5 * eiq)]

for nom in faits_bruts["noms"]:
    if types_de_colonnes(df)[nom] == "numerique":
        print(f"{nom:20s} {len(valeurs_aberrantes(df, nom)):>3d} valeur(s) aberrante(s)")

### À toi · exercice 4 ⭐⭐⭐ · Les alertes automatiques

Écris `alertes(df)` : elle renvoie une **liste de phrases** (une par problème détecté), et une liste vide si le fichier est sain. Quatre familles à couvrir, toutes sans nom de colonne en dur :

1. **colonne vide** (100 % de manquants) ou **très trouée** (plus de 30 % de manquants) ;
2. **colonne quasi-constante** : une seule valeur différente — elle n'apprend rien ;
3. **identifiant déguisé** : autant de valeurs différentes que de lignes ;
4. **valeurs aberrantes** : pour chaque colonne numérique, combien il y en a (utilise `valeurs_aberrantes`).

Et une dernière ligne si `df.duplicated()` trouve des doublons.

<details><summary>Indice</summary>

Une boucle `for c in df.columns:`, un `liste.append(...)` par cas, et `n = len(df)` pour les pourcentages. Attention : `nunique()` ignore déjà les manquants.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def alertes(df):
    """Ce qu'un analyste repère du coin de l'œil, en une liste de phrases."""
    liste, n = [], len(df)
    for c in df.columns:
        manquants, differentes = int(df[c].isna().sum()), int(df[c].nunique())
        if manquants == n:
            liste.append(f"« {c} » : colonne entièrement vide")
        elif manquants > 0.3 * n:
            liste.append(f"« {c} » : {manquants / n:.0%} de valeurs manquantes")
        if differentes == 1:
            liste.append(f"« {c} » : quasi-constante, une seule valeur")
        if differentes == n and n > 1:
            liste.append(f"« {c} » : identifiant déguisé, une valeur différente par ligne")
        if pd.api.types.is_numeric_dtype(df[c]) and len(valeurs_aberrantes(df, c)):
            liste.append(f"« {c} » : {len(valeurs_aberrantes(df, c))} valeurs aberrantes (1,5 × écart interquartile)")
    if int(df.duplicated().sum()):
        liste.append(f"{int(df.duplicated().sum())} ligne(s) strictement identique(s)")
    return liste

for a in alertes(df):
    print("⚠️ ", a)
```

</details>

In [ ]:
# À toi
def alertes(df):
    liste, n = [], len(df)
    # une boucle sur les colonnes, un append par problème détecté
    return liste

for a in alertes(df):
    print("⚠️ ", a)

In [ ]:
_sale = pd.DataFrame({"id": [1, 2, 3, 4], "constante": ["x"] * 4, "vide": [None] * 4, "n": [1, 1, 1, 99]})
verifier("Exercice 4 · une liste de phrases", lambda: isinstance(alertes(_sale), list) and all(isinstance(a, str) for a in alertes(_sale)))
verifier("Exercice 4 · colonne quasi-constante détectée", lambda: any("constante" in a for a in alertes(_sale)))
verifier("Exercice 4 · identifiant déguisé détecté", lambda: any("identifiant" in a for a in alertes(_sale)))
verifier("Exercice 4 · colonne vide détectée", lambda: any("vide" in a for a in alertes(_sale)))
verifier("Exercice 4 · valeur aberrante quantifiée", lambda: any("aberrantes" in a for a in alertes(_sale)))

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Replié comme une solution : il réinjecte l'implémentation de référence,
# pour que la suite du notebook fonctionne quand même.
if not alertes(_sale):
    def alertes(df):
        liste, n = [], len(df)
        for c in df.columns:
            manquants, differentes = int(df[c].isna().sum()), int(df[c].nunique())
            if manquants == n:
                liste.append(f"« {c} » : colonne entièrement vide")
            elif manquants > 0.3 * n:
                liste.append(f"« {c} » : {manquants / n:.0%} de valeurs manquantes")
            if differentes == 1:
                liste.append(f"« {c} » : quasi-constante, une seule valeur")
            if differentes == n and n > 1:
                liste.append(f"« {c} » : identifiant déguisé, une valeur différente par ligne")
            if pd.api.types.is_numeric_dtype(df[c]) and len(valeurs_aberrantes(df, c)):
                liste.append(f"« {c} » : {len(valeurs_aberrantes(df, c))} valeurs aberrantes (1,5 × écart interquartile)")
        if int(df.duplicated().sum()):
            liste.append(f"{int(df.duplicated().sum())} ligne(s) strictement identique(s)")
        return liste

## 4. Le rapport, puis les questions

### 4.1 · Le mini-rapport, rédigé à partir des faits

Le modèle n'a **jamais** accès au fichier. Il reçoit un dictionnaire de faits — des chiffres déjà calculés par ton code — et une seule mission : les mettre en phrases. C'est la différence entre un assistant qui rédige et un assistant qui invente.

In [ ]:
CONSIGNE_RAPPORT = """Tu es un analyste. Écris un rapport de 8 à 10 lignes en français à partir des faits ci-dessous.
Chaque chiffre du rapport doit venir des faits. N'invente aucun chiffre, ne fais aucune hypothèse.
Faits :
{faits}"""

def faits_du_fichier(df):
    """Le dictionnaire complet donné au modèle : rien d'autre ne sortira de sa plume."""
    base, types = decrire(df), types_de_colonnes(df)
    num = [c for c in df.columns if types[c] == "numerique"]
    pire = max(base["manquants"].items(), key=lambda kv: kv[1]) if base["manquants"] else ("aucune", 0)
    return {"lignes": base["lignes"], "colonnes": base["colonnes"],
            "numeriques": len(num), "categorielles": base["colonnes"] - len(num),
            "manquants_total": int(sum(base["manquants"].values())),
            "pire_manquants": [str(pire[0]), int(pire[1])], "doublons": base["doublons"],
            "colonnes_detail": {c: stats_colonne(df, c) for c in num[:4]},
            "nb_alertes": len(alertes(df))}

def rediger_rapport(faits):
    """Fait rédiger le rapport à partir des SEULS faits fournis."""
    return demander("Écris le rapport.", systeme=CONSIGNE_RAPPORT.format(faits=json.dumps(faits, ensure_ascii=False)))

mes_alertes = alertes(df)
faits = faits_du_fichier(df)
rapport = rediger_rapport(faits)
print(len(mes_alertes), "alerte(s) sur", JEU, "— un fichier propre en donne zéro, c'est un résultat aussi\n")
for a in mes_alertes:
    print("⚠️ ", a)
print(rapport)

### À toi · exercice 5 ⭐⭐ · Aucun chiffre inventé

Une consigne n'est pas une garantie. Un modèle qui reçoit « n'invente aucun chiffre » en invente quand même, surtout un petit modèle. La seule façon de le savoir, c'est de **vérifier mécaniquement** : tout nombre écrit dans le rapport doit se retrouver dans le dictionnaire de faits.

Écris deux fonctions :
- `nombres(texte)` : la liste de tous les nombres écrits dans un texte, sous forme de chaînes (`"342"`, `"4201.75"`) ;
- `verifier_chiffres(rapport, faits)` : la liste des nombres du rapport **absents** des faits. Une liste vide = rapport honnête.

Astuce : `json.dumps(faits)` transforme les faits en un grand texte — il suffit d'y chercher les mêmes nombres.

<details><summary>Indice</summary>

`re.findall(r"-?\d+(?:[.,]\d+)?", texte)`. Puis compare les deux listes avec un `set`, en remplaçant la virgule décimale par un point.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def nombres(texte):
    """Tous les nombres écrits dans un texte, sous forme de chaînes."""
    return re.findall(r"-?\d+(?:[.,]\d+)?", texte)

def verifier_chiffres(rapport, faits):
    """Les nombres du rapport qui ne viennent PAS du dictionnaire de faits."""
    connus = set(nombres(json.dumps(faits, ensure_ascii=False)))
    return [n for n in nombres(rapport) if n.replace(",", ".") not in connus]

print("chiffres inventés :", verifier_chiffres(rapport, faits))
```

</details>

In [ ]:
# À toi
def nombres(texte):
    return []

def verifier_chiffres(rapport, faits):
    return None

print("chiffres inventés :", verifier_chiffres(rapport, faits))

In [ ]:
verifier("Exercice 5 · nombres() trouve entiers et décimaux", lambda: nombres("342 lignes, moyenne 4201.75") == ["342", "4201.75"])
verifier("Exercice 5 · le rapport du modèle ne contient rien d'inventé", lambda: verifier_chiffres(rapport, faits) == [])
verifier("Exercice 5 · un chiffre inventé est repéré", lambda: verifier_chiffres("Le poids moyen est de 9999 g.", faits) == ["9999"])
verifier("Exercice 5 · un chiffre des faits n'est pas signalé", lambda: verifier_chiffres(f"Il y a {faits['lignes']} lignes.", faits) == [])

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Replié comme une solution : il réinjecte l'implémentation de référence,
# pour que la suite du notebook fonctionne quand même.
if verifier_chiffres(rapport, faits) is None:
    def nombres(texte):
        return re.findall(r"-?\d+(?:[.,]\d+)?", texte)
    def verifier_chiffres(rapport, faits):
        connus = set(nombres(json.dumps(faits, ensure_ascii=False)))
        return [n for n in nombres(rapport) if n.replace(",", ".") not in connus]

Le contrôle en action. On prend le vrai rapport, on y glisse **une** phrase inventée — exactement le genre de phrase qu'un modèle ajoute pour faire joli — et on regarde si le contrôle la voit.

In [ ]:
rapport_truque = rapport + "\nLe fichier a été collecté en 2019 auprès de 512 individus, pour un coût moyen de 37.5 euros."

print("rapport du modèle   :", verifier_chiffres(rapport, faits), "→ aucun chiffre inventé" if not verifier_chiffres(rapport, faits) else "")
print("rapport avec un ajout:", verifier_chiffres(rapport_truque, faits))
print("\nEn mode démo, le faux modèle recopie les faits : il ne peut pas inventer.")
print("Avec USE_MODEL = True, relance cette cellule : c'est là que la liste se remplit vraiment.")

### 4.2 · Poser une question en français

Le rapport dit ce qu'il y a dans le fichier. Il ne répond pas à **ta** question. Pour ça, il faudrait que l'agent écrive du code pandas et l'exécute — et c'est exactement le moment où un projet d'agent devient dangereux.

**Pourquoi pas `exec` ?** Parce qu'un modèle qui écrit du Python libre peut écrire `df.to_csv("/content/fuite.csv")`, `import os`, `open(...)`, ou simplement une boucle infinie. Aucune consigne en langage naturel ne l'en empêche de façon fiable : une consigne est une *suggestion*, pas une barrière.

**La barrière, c'est une liste blanche.** On définit un mini-langage de six opérations, et notre code n'exécute que celles-là, avec un `if` par opération. Tout ce qui n'est pas dans la liste est refusé — pas « analysé », pas « nettoyé » : refusé.

| Commande | Ce qu'elle fait |
|---|---|
| `moyenne(colonne)` | la moyenne d'une colonne numérique |
| `mediane(colonne)` | sa médiane |
| `compte(colonne)` | le nombre de valeurs renseignées |
| `top(colonne, n)` | les `n` valeurs les plus fréquentes (un classement) |
| `moyenne_par(groupe, colonne)` | la moyenne de `colonne` pour chaque valeur de `groupe` (un classement) |
| `filtre(colonne, >, valeur) puis moyenne(colonne)` | on filtre d'abord, on calcule ensuite |

Trois briques, dans cet ordre : **lire** une commande sans l'exécuter, **contrôler** qu'elle est dans la liste blanche (l'exercice 6), puis l'**exécuter** avec un `if` par opération — jamais d'`exec`, jamais d'`eval`. Le pire qu'une commande puisse alors faire, c'est renvoyer un mauvais chiffre : pas toucher au disque.

In [ ]:
OPERATIONS = ("moyenne", "mediane", "compte", "top", "moyenne_par", "filtre")
MOTIF = re.compile(r"^([a-z_]+)\(([^()]*)\)$")

def analyser_commande(commande):
    """'filtre(a, >, 3) puis moyenne(b)' → [('filtre', ['a','>','3']), ('moyenne', ['b'])]. None si illisible."""
    etapes = []
    for morceau in str(commande).strip().split(" puis "):
        trouve = MOTIF.match(morceau.strip())
        if not trouve:
            return None
        etapes.append((trouve.group(1), [a.strip() for a in trouve.group(2).split(",") if a.strip()]))
    return etapes

COLS = list(df.columns)
for essai in ["moyenne(body_mass_g)", "filtre(bill_length_mm, >, 45) puis moyenne(body_mass_g)", "df.to_csv('fuite.csv')"]:
    print(f"{essai:55s} → {analyser_commande(essai)}")

### À toi · exercice 6 ⭐⭐⭐ · La liste blanche

`analyser_commande` sait *lire* une commande. Elle ne dit pas si on a le droit de l'exécuter : `supprimer(species)` se lit très bien.

Écris `commande_autorisee(commande, colonnes)` qui renvoie `True` seulement si **tout** est en règle :

1. la commande se lit (`analyser_commande` ne renvoie pas `None`) ;
2. chaque opération est dans `OPERATIONS` ;
3. chaque argument est soit un **nom de colonne existant**, soit un **nombre**, soit l'un des signes `>` `<`.

Tout le reste est refusé : `df.to_csv(...)`, `import os`, `moyenne(__class__)`, `supprimer(species)`, `moyenne(colonne_qui_nexiste_pas)`.

<details><summary>Indice</summary>

Un argument est un nombre si `arg.replace(".", "", 1).isdigit()`. Une boucle `for op, args in etapes:` avec un `return False` dès qu'une règle est violée.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def commande_autorisee(commande, colonnes):
    """Liste blanche : opérations connues, arguments = colonnes existantes, nombres ou signes."""
    etapes = analyser_commande(commande)
    if not etapes:
        return False
    for op, args in etapes:
        if op not in OPERATIONS or not args:
            return False
        for arg in args:
            nombre = arg.replace(".", "", 1).isdigit()
            if arg not in colonnes and not nombre and arg not in (">", "<"):
                return False
    return True

for essai in ["moyenne(body_mass_g)", "df.to_csv('fuite.csv')", "import os", "moyenne(__class__)"]:
    print(commande_autorisee(essai, list(df.columns)), "·", essai)
```

</details>

In [ ]:
# À toi
def commande_autorisee(commande, colonnes):
    return None

for essai in ["moyenne(" + COLS[-1] + ")", "df.to_csv('fuite.csv')", "import os", "moyenne(__class__)"]:
    print(commande_autorisee(essai, COLS), "·", essai)

In [ ]:
_num = [c for c in COLS if types_de_colonnes(df)[c] == "numerique"][0]
verifier("Exercice 6 · une commande légitime passe", lambda: commande_autorisee(f"moyenne({_num})", COLS) is True)
verifier("Exercice 6 · deux étapes légitimes passent", lambda: commande_autorisee(f"filtre({_num}, >, 10) puis moyenne({_num})", COLS) is True)
verifier("Exercice 6 · l'écriture de fichier est refusée", lambda: commande_autorisee("df.to_csv('fuite.csv')", COLS) is False)
verifier("Exercice 6 · un import est refusé", lambda: commande_autorisee("import os", COLS) is False)
verifier("Exercice 6 · les doubles soulignés sont refusés", lambda: commande_autorisee("moyenne(__class__)", COLS) is False)
verifier("Exercice 6 · une opération inconnue est refusée", lambda: commande_autorisee(f"supprimer({_num})", COLS) is False)
verifier("Exercice 6 · une colonne inexistante est refusée", lambda: commande_autorisee("moyenne(colonne_inventee)", COLS) is False)

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Replié comme une solution : il réinjecte l'implémentation de référence,
# pour que la suite du notebook fonctionne quand même.
if commande_autorisee(f"moyenne({_num})", COLS) is not True:
    def commande_autorisee(commande, colonnes):
        etapes = analyser_commande(commande)
        if not etapes:
            return False
        for op, args in etapes:
            if op not in OPERATIONS or not args:
                return False
            for arg in args:
                if arg not in colonnes and not arg.replace(".", "", 1).isdigit() and arg not in (">", "<"):
                    return False
        return True

In [ ]:
def executer(df, commande):
    """Exécute une commande du mini-langage. Un if par opération, jamais d'exec ni d'eval."""
    table, resultat = df, None
    for op, args in analyser_commande(commande) or []:
        if op == "filtre":
            col, signe, valeur = args[0], args[1], float(args[2])
            table = table[table[col] > valeur] if signe == ">" else table[table[col] < valeur]
        elif op == "moyenne":
            resultat = float(table[args[0]].mean())
        elif op == "mediane":
            resultat = float(table[args[0]].median())
        elif op == "compte":
            resultat = int(table[args[0]].count())
        elif op == "top":
            resultat = table[args[0]].value_counts().head(int(args[1]) if len(args) > 1 else 5)
        elif op == "moyenne_par":
            resultat = table.groupby(args[0])[args[1]].mean().sort_values(ascending=False)
    return resultat

print(executer(df, f"moyenne({_num})"))
print(executer(df, f"filtre({_num}, >, 0) puis mediane({_num})"))

Et le contrôle du **résultat**, avant de le commenter. Deux questions, dans cet ordre :

1. **Le résultat est-il exploitable ?** Un `NaN` est un piège : il s'affiche, il se met en phrase, et il ne veut rien dire. Une Series vide aussi.
2. **A-t-il la forme qu'attend la question ?** « Quel est le poids moyen ? » attend **un nombre**. « Quelle espèce est la plus lourde ? » attend **un classement**. Un nombre en réponse à la deuxième question, c'est une réponse à une autre question — et c'est le piège central de cette séance.

In [ ]:
def resultat_valide(resultat):
    """Un nombre fini, ou une Series non vide et sans NaN. Tout le reste est refusé."""
    if isinstance(resultat, (int, float, np.integer, np.floating)):
        return bool(np.isfinite(resultat))
    if isinstance(resultat, pd.Series):
        return len(resultat) > 0 and not resultat.isna().any()
    return False

def forme_obtenue(resultat):
    return "classement" if isinstance(resultat, pd.Series) else "nombre"

def forme_attendue(question):
    """Un superlatif ou un « par ... » désigne un élément : la question attend un classement, pas un nombre."""
    q = question.lower()
    if re.search(r"\b(le|la|les) plus\b", q) or "laquelle" in q or "lequel" in q or " par " in q:
        return "classement"
    return "nombre"

for q in ["Quel est le poids moyen des manchots ?", "Quelle espèce de manchot est la plus lourde ?"]:
    print(forme_attendue(q), "←", q)
print(resultat_valide(float("nan")), resultat_valide(3.5), resultat_valide(pd.Series(dtype=float)))

L'agent complet, enfin. Cinq étapes, et un refus possible à chacune : **écrire** la requête → **contrôler** qu'elle est autorisée → **exécuter** → **vérifier** la forme du résultat → **commenter** en une phrase.

Remarque le prompt : le modèle reçoit la liste des colonnes et la liste des opérations, jamais les données. Il ne peut donc pas inventer un chiffre — au pire, il écrit une mauvaise requête.

In [ ]:
CONSIGNE_REQUETE = """Tu es un analyste. Écris la requête qui répond à la question, et RIEN d'autre.
Opérations autorisées, aucune autre : moyenne(colonne) · mediane(colonne) · compte(colonne) ·
top(colonne, n) · moyenne_par(groupe, colonne) · filtre(colonne, >, valeur) puis moyenne(colonne)
Colonnes numériques : {num}
Colonnes catégorielles : {cat}
Écris la requête sur une seule ligne."""

CONSIGNE_COMMENT = """Tu commentes un résultat de calcul en UNE phrase française.
Interdiction d'écrire un chiffre qui n'est pas dans les faits ci-dessous.
Question : {question}
Faits :
{faits}"""

def ecrire_requete(question, df):
    """Fait écrire la requête par le modèle : il voit les noms des colonnes, jamais les données."""
    types = types_de_colonnes(df)
    systeme = CONSIGNE_REQUETE.format(num=", ".join(c for c in df.columns if types[c] == "numerique"),
                                      cat=", ".join(c for c in df.columns if types[c] == "categorielle"))
    return (demander(question, systeme=systeme).strip().splitlines() or [""])[0].strip()

def commenter(question, commande, resultat):
    """Une phrase, écrite à partir des SEULS chiffres calculés par le code."""
    if isinstance(resultat, pd.Series):
        faits_r = {"classement": [[str(i), round(float(v), 2)] for i, v in resultat.head(5).items()],
                   "nb_groupes": int(len(resultat))}
    else:
        faits_r = {"valeur": round(float(resultat), 2), "commande": commande}
    return demander("Commente le résultat.",
                    systeme=CONSIGNE_COMMENT.format(question=question, faits=json.dumps(faits_r, ensure_ascii=False)))

In [ ]:
REFUS = {"commande": "Requête refusée : elle sort de la liste blanche.",
         "resultat": "Résultat inexploitable (vide, NaN ou colonne absente) — je ne commente pas.",
         "forme":    "Cette requête est valide mais elle répond à une AUTRE question que celle posée."}

def repondre_a(question, df):
    """Question en français → requête → contrôle → exécution → vérification de forme → une phrase."""
    fiche = {"question": question, "commande": ecrire_requete(question, df), "resultat": None, "phrase": ""}
    if not commande_autorisee(fiche["commande"], list(df.columns)):
        fiche["phrase"] = REFUS["commande"]
        return fiche
    try:
        fiche["resultat"] = executer(df, fiche["commande"])
    except Exception as e:
        fiche["phrase"] = f"{REFUS['resultat']} ({type(e).__name__})"
        return fiche
    if not resultat_valide(fiche["resultat"]):
        fiche["phrase"] = REFUS["resultat"]
    elif forme_obtenue(fiche["resultat"]) != forme_attendue(question):
        fiche["phrase"] = REFUS["forme"]
    else:
        fiche["phrase"] = commenter(question, fiche["commande"], fiche["resultat"])
    return fiche

In [ ]:
QUESTIONS = {
    "manchots": ["Quel est le poids moyen des manchots ?", "Quel est le poids moyen par espèce ?",
                 "Quelles sont les îles les plus fréquentes ?"],
    "pourboires": ["Quel est le pourboire moyen ?", "Quel est le pourboire moyen par jour ?",
                   "Quels sont les jours les plus fréquents ?"],
    "pokemon": ["Quelle est l'attaque moyenne ?", "Quelle est l'attaque moyenne par type ?",
                "Quels sont les types les plus fréquents ?"],
}
for question in QUESTIONS.get(JEU, QUESTIONS["manchots"]):
    fiche = repondre_a(question, df)
    print("❓", question)
    print("   requête :", fiche["commande"])
    print("   réponse :", fiche["phrase"], "\n")

vol = "df.to_csv('/content/fuite.csv')"          # ce qu'un modèle bavard finit toujours par proposer
print("❓ requête hors liste blanche :", vol)
print("   réponse :", REFUS["commande"] if not commande_autorisee(vol, list(df.columns)) else "acceptée — le garde-fou a sauté !")

### 4.3 · Le piège : une requête juste qui répond à côté

Voici la question qui a fait échouer plus d'un projet d'agent en entreprise :

> **« Quelle espèce de manchot est la plus lourde ? »**

Elle demande **un nom d'espèce**. Regarde ce que le modèle en fait.

In [ ]:
piege = "Quelle espèce de manchot est la plus lourde ?" if JEU == "manchots" else "Quel jour est le plus généreux ?"
commande_piege = ecrire_requete(piege, df)
autorisee = commande_autorisee(commande_piege, list(df.columns))
resultat_piege = executer(df, commande_piege) if autorisee else None

print("question        :", piege)
print("requête écrite  :", commande_piege, "· autorisée :", autorisee)
print("elle s'exécute  :", resultat_piege, "· résultat valide :", resultat_valide(resultat_piege))
print(">>> la phrase SANS garde-fou de forme :")
print("   ", commenter(piege, commande_piege, resultat_piege) if resultat_valide(resultat_piege)
      else "(requête refusée dès la liste blanche)")

print("\n>>> avec le garde-fou de forme :", repondre_a(piege, df)["phrase"])

bonne = "Quel est le poids moyen par espèce ?" if JEU == "manchots" else "Quel est le pourboire moyen par jour ?"
fiche_bonne = repondre_a(bonne, df)
print("\nquestion reformulée :", bonne)
print("requête             :", fiche_bonne["commande"])
print("réponse             :", fiche_bonne["phrase"])

Tout est vert. La requête est dans la liste blanche, elle s'exécute sans erreur, le résultat est un nombre fini, et la phrase est du français impeccable. **Et pourtant la réponse est fausse** : on a demandé *quelle espèce*, la machine a répondu *le poids moyen de tout le monde*. Aucun contrôle technique ne peut le voir — le code est correct, il répond juste à une autre question.

C'est le seul type de bug qu'un agent d'analyse produit en masse, et le seul qu'aucun test unitaire n'attrape. En mode démo, on l'a codé exprès dans le faux modèle (la branche `"plus lourd"` de `_requete_factice`), parce qu'un vrai petit modèle le produit spontanément mais pas systématiquement : le forcer permet de le regarder en face.

Le seul garde-fou qui l'attrape en partie, c'est celui de la **forme** : la question désigne un élément, donc elle attend un classement ; on a reçu un nombre. L'agent ne corrige pas la requête — il refuse de commenter, ce qui est déjà beaucoup mieux que de mentir avec assurance. La question reformulée, elle, passe.

Deux enseignements, et ce sont les deux à retenir du projet :

1. **Le garde-fou de forme attrape le symptôme, pas la cause.** Il empêche l'agent de raconter n'importe quoi, il ne lui fait pas écrire la bonne requête. Entre « je refuse de répondre » et « voici une réponse fausse et fluide », le refus gagne toujours — mais ce n'est pas une réponse.
2. **La requête écrite doit rester lisible par un humain.** C'est tout l'intérêt du mini-langage : `moyenne(body_mass_g)` face à `moyenne_par(species, body_mass_g)`, l'erreur saute aux yeux en une seconde. Avec dix lignes de pandas générées, personne ne relit — et le bug passe en production.

## 5. Les trois graphiques proposés

Trois graphiques au hasard n'apprennent rien. Trois graphiques **choisis par une règle** apprennent quelque chose, et surtout : la règle se discute, alors qu'un hasard ne se discute pas.

La règle codée ici tient en trois lignes :

| Proposition | Colonnes choisies | Ce qu'on voit |
|---|---|---|
| **distribution** | la colonne numérique la plus variée | la forme des valeurs : un pic, deux bosses, une traîne |
| **comparaison** | une colonne catégorielle de 2 à 10 valeurs × une colonne numérique | un écart entre groupes |
| **relation** | les deux colonnes numériques les plus corrélées | un lien, ou son absence |

### À toi · exercice 7 ⭐⭐ · La règle de choix

Écris `proposer_graphiques(df)` : elle renvoie une liste de dictionnaires `{"type", "colonnes", "pourquoi"}` — au plus un de chaque type, et seulement si le fichier s'y prête.

- `"distribution"` : la première colonne numérique ayant plus de 5 valeurs différentes ;
- `"comparaison"` : la première colonne catégorielle ayant entre 2 et 10 valeurs différentes, croisée avec la colonne numérique de la distribution ;
- `"relation"` : la paire de colonnes numériques la plus corrélée (`meilleure_paire` ci-dessous), s'il y en a au moins deux.

Le champ `"pourquoi"` est une phrase qui justifie le choix : c'est ce que l'humain lira avant de dire oui ou non.

<details><summary>Indice</summary>

Construis d'abord `num` et `cat` par compréhension de liste sur `df.columns`, avec `types_de_colonnes(df)` et `df[c].nunique()`. Puis trois `if` et trois `append`.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def proposer_graphiques(df):
    """Trois propositions choisies par le type des colonnes — jamais au hasard."""
    types = types_de_colonnes(df)
    num = [c for c in df.columns if types[c] == "numerique" and df[c].nunique() > 5]
    cat = [c for c in df.columns if types[c] == "categorielle" and 2 <= df[c].nunique() <= 10]
    propositions = []
    if num:
        propositions.append({"type": "distribution", "colonnes": [num[0]],
                             "pourquoi": f"« {num[0] } » est numérique et variée : voir la forme de ses valeurs"})
    if num and cat:
        propositions.append({"type": "comparaison", "colonnes": [cat[0], num[0]],
                             "pourquoi": f"« {cat[0]} » a peu de catégories : comparer « {num[0]} » entre elles"})
    if len(num) >= 2:
        a, b = meilleure_paire(df, num)
        propositions.append({"type": "relation", "colonnes": [a, b],
                             "pourquoi": f"« {a} » et « {b} » sont les deux colonnes numériques les plus corrélées"})
    return propositions

propositions = proposer_graphiques(df)
```

</details>

In [ ]:
def meilleure_paire(df, colonnes):
    """Les deux colonnes numériques les plus corrélées (en valeur absolue)."""
    m = df[colonnes].corr().abs().fillna(0).to_numpy(copy=True)
    np.fill_diagonal(m, 0)
    i, j = np.unravel_index(np.argmax(m), m.shape)
    return colonnes[i], colonnes[j]

# À toi
def proposer_graphiques(df):
    return []

propositions = proposer_graphiques(df)
for p in propositions:
    print(p["type"], "·", p["colonnes"], "·", p["pourquoi"])

In [ ]:
verifier("Exercice 7 · trois propositions", lambda: len(propositions) == 3)
verifier("Exercice 7 · trois types différents", lambda: len({p["type"] for p in propositions}) == 3)
verifier("Exercice 7 · les colonnes existent", lambda: all(c in df.columns for p in propositions for c in p["colonnes"]))
verifier("Exercice 7 · chaque proposition est justifiée", lambda: all(len(p["pourquoi"]) > 20 for p in propositions))

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Replié comme une solution : il réinjecte l'implémentation de référence,
# pour que la suite du notebook fonctionne quand même.
if len(propositions) < 3:
    def proposer_graphiques(df):
        types = types_de_colonnes(df)
        num = [c for c in df.columns if types[c] == "numerique" and df[c].nunique() > 5]
        cat = [c for c in df.columns if types[c] == "categorielle" and 2 <= df[c].nunique() <= 10]
        p = []
        if num:
            p.append({"type": "distribution", "colonnes": [num[0]],
                      "pourquoi": f"« {num[0]} » est numérique et variée : voir la forme de ses valeurs"})
        if num and cat:
            p.append({"type": "comparaison", "colonnes": [cat[0], num[0]],
                      "pourquoi": f"« {cat[0]} » a peu de catégories : comparer « {num[0]} » entre elles"})
        if len(num) >= 2:
            a, b = meilleure_paire(df, num)
            p.append({"type": "relation", "colonnes": [a, b],
                      "pourquoi": f"« {a} » et « {b} » sont les deux colonnes numériques les plus corrélées"})
        return p
    propositions = proposer_graphiques(df)

In [ ]:
def tracer(df, proposition):
    """Trace une proposition : distribution, comparaison ou relation. Titre et axes toujours nommés."""
    cols = proposition["colonnes"]
    plt.figure(figsize=(6, 3.4))
    if proposition["type"] == "distribution":
        plt.hist(df[cols[0]].dropna(), bins=20, color="tab:blue", edgecolor="white")
        plt.xlabel(cols[0])
        plt.ylabel("nombre de lignes")
        plt.title(f"Distribution de {cols[0]}")
    elif proposition["type"] == "comparaison":
        moyennes = df.groupby(cols[0])[cols[1]].mean().sort_values()
        plt.barh(moyennes.index.astype(str), moyennes.values, color="tab:green")
        plt.xlabel(f"{cols[1]} (moyenne)")
        plt.ylabel(cols[0])
        plt.title(f"{cols[1]} moyen par {cols[0]}")
    else:
        plt.scatter(df[cols[0]], df[cols[1]], s=12, alpha=0.6, color="tab:orange")
        plt.xlabel(cols[0])
        plt.ylabel(cols[1])
        plt.title(f"{cols[1]} en fonction de {cols[0]}")
    plt.tight_layout()
    plt.show()

print("Prêt à tracer — mais rien n'est tracé tant que tu n'as pas dit oui.")

### La validation humaine : la liste `DECISIONS`

L'agent a proposé. C'est ton tour. Une décision par proposition, dans l'ordre, **trois formes possibles** :

- `"oui"` → le graphique est tracé ;
- `"non : mon motif"` → rien n'est tracé, et le motif part au journal ;
- une **consigne** contenant `plutôt` → l'agent remplace la proposition par la tienne. Format : `plutôt distribution de <colonne>`, `plutôt comparaison de <colonne> par <colonne>`, `plutôt relation de <colonne> et <colonne>`.

Édite la liste ci-dessous, relance la cellule, et le résultat change. Pas d'`input()`, pas de widget : ton choix est **écrit dans le notebook**, donc relisible et reproductible six mois plus tard.

In [ ]:
AUTRE = [c for c in df.columns if types_de_colonnes(df)[c] == "numerique"][1:2]     # 2e colonne numérique, s'il y en a une

DECISIONS = [                       # ← une décision par proposition, dans l'ordre. Édite, puis relance.
    "oui",
    "oui",
    f"non, les deux colonnes du nuage sont liées par construction — plutôt distribution de {AUTRE[0] if AUTRE else ''}",
]
print(len(DECISIONS), "décisions écrites pour", len(propositions), "propositions")

In [ ]:
def consigne_en_proposition(consigne, df):
    """« plutôt distribution de X » → une proposition du même format que celles de l'agent."""
    apres = consigne.split("plutôt", 1)[1]
    type_ = next((t for t in ("distribution", "comparaison", "relation") if t in apres), "distribution")
    colonnes = [c for c in df.columns if c.lower() in apres.lower()]
    return {"type": type_, "colonnes": colonnes, "pourquoi": "demandé par l'humain : " + consigne.strip()}

JOURNAL = []
for i, proposition in enumerate(propositions):
    decision = DECISIONS[i] if i < len(DECISIONS) else "non : pas de décision écrite"
    retenue, statut = proposition, "refusé"
    if "plutôt" in decision.lower():
        retenue, statut = consigne_en_proposition(decision, df), "modifié"
    elif decision.strip().lower().startswith("oui"):
        statut = "accepté"
    JOURNAL.append({"proposition": f"{proposition['type']} {proposition['colonnes']}", "decision": decision,
                    "statut": statut, "trace": f"{retenue['type']} {retenue['colonnes']}" if statut != "refusé" else "—"})
    print(f"[{statut}] {proposition['type']} {proposition['colonnes']}")
    if statut != "refusé" and retenue["colonnes"]:
        print("   →", retenue["pourquoi"])
        tracer(df, retenue)

## 6. Le journal, le banc de test, la fiche projet

### Le journal des décisions

Un agent qui demande l'autorisation ne sert à rien si personne ne peut relire ce qui a été autorisé. Le journal est la pièce qu'on montre : trois lignes, et on sait ce que la machine a proposé, ce qu'un humain en a fait, et ce qui a fini tracé.

In [ ]:
print("proposé par l'agent".ljust(46), "statut".ljust(9), "finalement tracé")
print("-" * 100)
for ligne in JOURNAL:
    print(ligne["proposition"][:45].ljust(46), ligne["statut"].ljust(9), ligne["trace"])
print()
for ligne in JOURNAL:
    if ligne["statut"] != "accepté":
        print("motif :", ligne["decision"])
print(f"\n{sum(1 for l in JOURNAL if l['statut'] == 'accepté')} accepté(s), "
      f"{sum(1 for l in JOURNAL if l['statut'] == 'modifié')} modifié(s), "
      f"{sum(1 for l in JOURNAL if l['statut'] == 'refusé')} refusé(s)")

### Le banc de test

Six questions, une requête attendue pour chacune. On ne juge pas la phrase finale — on juge la **requête écrite par le modèle**, parce que c'est là que tout se joue. La dernière ligne est le piège de la section 4.3 : elle **doit** échouer.

In [ ]:
BANC = [   # écrit pour les manchots — réécris ces lignes pour ton fichier
    {"question": "Quel est le poids moyen des manchots ?", "attendu": "moyenne(body_mass_g)"},
    {"question": "Quel est le poids médian des manchots ?", "attendu": "mediane(body_mass_g)"},
    {"question": "Quelles sont les îles les plus fréquentes ?", "attendu": "top(island, 5)"},
    {"question": "Quel est le poids moyen par espèce ?", "attendu": "moyenne_par(species, body_mass_g)"},
    {"question": "Combien de manchots ont un poids renseigné ?", "attendu": "compte(body_mass_g)"},
    {"question": "Quelle espèce de manchot est la plus lourde ?", "attendu": "moyenne_par(species, body_mass_g)"},
]

def evaluer(banc, df):
    """Compare la requête écrite par le modèle à la requête attendue."""
    resultats = []
    for cas in banc:
        obtenue = ecrire_requete(cas["question"], df)
        ok = obtenue == cas["attendu"]
        resultats.append(ok)
        print(("✅" if ok else "❌"), cas["question"])
        print(f"     attendu : {cas['attendu']}\n     obtenu  : {obtenue}")
    print(f"\n=== {sum(resultats)}/{len(resultats)} requêtes correctes ===")
    return resultats

scores_banc = evaluer(BANC, df)

**L'échec à analyser.** La dernière ligne échoue, et c'est voulu : *« Quelle espèce de manchot est la plus lourde ? »* produit `moyenne(body_mass_g)` au lieu de `moyenne_par(species, body_mass_g)`. La requête est valide, elle s'exécute, elle rend un nombre fini. Elle répond simplement à une autre question.

Ce que le banc de test montre, et qui est la conclusion du projet : **les cinq premières lignes ne prouvent rien sur la sixième**. Un agent qui réussit 5/6 sur des questions simples échouera exactement de la même manière sur la première question réellement intéressante — celle qui demande de désigner un élément, pas de sortir un agrégat. Le taux global (83 %) est trompeur ; c'est la nature de l'échec qui compte, pas son nombre.

Si tu travailles sur ton propre fichier : réécris ces six lignes avec tes colonnes, et garde-en une qui échoue. Un banc de test où tout passe ne t'apprend rien.

### Le même agent sur d'autres fichiers

La contrainte du projet était de ne jamais écrire un nom de colonne en dur. On la vérifie de la seule façon honnête : en passant les trois fichiers dans la même chaîne, sans rien changer d'autre. Le troisième (Pokémon) est le fichier « difficile » : une colonne identifiant, une colonne à moitié vide, des noms de colonnes avec espaces et ponctuation.

Regarde la dernière colonne du résultat : sur Pokémon, l'agent propose **deux** graphiques au lieu de trois. Aucune colonne de texte n'y a entre 2 et 10 valeurs différentes, donc la règle « comparaison » ne s'applique pas. Il ne plante pas et il n'invente pas un graphique pour faire le compte : il en propose moins. C'est le comportement qu'on veut d'une règle de choix — et c'est pour ça qu'on l'a écrite en trois lignes lisibles plutôt que de laisser le modèle choisir.

In [ ]:
for nom in JEUX:
    try:
        autre = pd.read_csv(JEUX[nom])
    except Exception:
        print(f"{nom:12s} : réseau indisponible, on passe")
        continue
    f = faits_du_fichier(autre)
    print(f"{nom:12s} : {f['lignes']:>4d} lignes × {f['colonnes']:>2d} colonnes · "
          f"{f['manquants_total']:>4d} manquants · {f['nb_alertes']} alerte(s) · "
          f"{len(proposer_graphiques(autre))} graphique(s) proposé(s)")
    print("             ", rediger_rapport(f).splitlines()[0])
    print("             ", (alertes(autre) or ["aucune alerte"])[0])

## Conclusion et fiche projet

À retenir :

- **Des faits, puis des phrases.** Tous les chiffres sortent de pandas et passent par un dictionnaire ; le modèle ne fait que rédiger. C'est ce qui rend `verifier_chiffres` possible — et un rapport invérifiable ne vaut rien.
- **Une liste blanche, jamais un `exec`.** Six opérations, un `if` par opération, un refus propre pour tout le reste. Le mini-langage a un second mérite : la requête reste **relisible en une seconde** par un humain.
- **Vérifier la forme avant de commenter.** Un `NaN` se met en phrase aussi bien qu'un vrai chiffre. Un nombre en réponse à une question qui attend un nom, aussi.
- **Le vrai risque n'est pas le code faux, c'est le code juste qui répond à côté.** Aucun test technique ne l'attrape ; seule la relecture de la requête par quelqu'un qui connaît la question le fait.
- **L'agent propose, l'humain dispose.** Le journal des décisions n'est pas de la paperasse : c'est la seule trace de qui a validé quoi.

Remplis la fiche ci-dessous : c'est ce que tu montreras.

In [ ]:
MA_SYNTHESE = """(à remplir) J'ai fait tourner l'agent sur ... . Il a bien vu ... ,
il a raté ... . Le graphique que j'ai refusé était ... , je l'ai remplacé par ... parce que ... .
Ce qu'un humain apporte encore ici : ... ."""

print("=== FICHE PROJET B3 · L'ANALYSTE AUTOMATIQUE ===")
print(f"Fichier            : {JEU} · {faits['lignes']} lignes × {faits['colonnes']} colonnes")
print(f"Alertes            : {len(mes_alertes)} · manquants : {faits['manquants_total']} · doublons : {faits['doublons']}")
print(f"Rapport            : {len(rapport.splitlines())} lignes · chiffres inventés : {len(verifier_chiffres(rapport, faits))}")
print(f"Graphiques         : {sum(1 for l in JOURNAL if l['statut'] == 'accepté')} acceptés, "
      f"{sum(1 for l in JOURNAL if l['statut'] == 'modifié')} modifiés, "
      f"{sum(1 for l in JOURNAL if l['statut'] == 'refusé')} refusés (sur {len(propositions)} proposés)")
print(f"Banc de test       : {sum(scores_banc)}/{len(scores_banc)} requêtes correctes")
print(f"Mode               : {'modèle Qwen2.5-0.5B' if USE_MODEL else 'démo (llm_factice)'}")
print("\nMa synthèse :", MA_SYNTHESE)

## Pour aller plus loin

- **Compare avec `ydata-profiling`** (`!pip install ydata-profiling`, puis `ProfileReport(df).to_notebook_iframe()`) : lui sort cent pages, toi dix lignes. Laquelle des deux sorties utiliserais-tu vraiment un lundi matin, et pourquoi ? La réponse est moins évidente qu'elle en a l'air.
- **Fais écrire le code du graphique au modèle** plutôt que de le choisir dans une liste, et ne l'exécute qu'après relecture affichée. C'est exactement le débat « l'agent qui exécute du code » en entreprise : compare le risque et le gain avec la version liste blanche de ce notebook.
- **Ajoute une étape « question métier »** : l'agent propose trois questions auxquelles ce fichier peut répondre, tu en choisis une dans `DECISIONS`, il produit le graphique correspondant.
- **Élargis le mini-langage** avec `somme`, `ecart_type`, `filtre(colonne, ==, valeur)` — et note combien de lignes de garde-fou chaque nouvelle opération coûte. C'est le vrai prix d'un agent qui exécute du code.

Liens utiles : le modèle https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct · pandas en 10 minutes https://pandas.pydata.org/docs/user_guide/10min.html · `describe()` https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html · galerie matplotlib https://matplotlib.org/stable/plot_types/index.html · cartes de chaleur seaborn https://seaborn.pydata.org/generated/seaborn.heatmap.html · ce qu'est un agent en 40 pages https://www.kaggle.com/whitepaper-agents